# 3.4: Notebook — Feature preprocessing: Scaling, encoding and imputation

In this notebook, we bring together the three core preprocessing techniques from Lesson 3 and apply them to a single realistic dataset. You will see how each technique works, why it is needed, and how to apply each one correctly after splitting your data.

## Learning objectives
- Handle missing values correctly using SimpleImputer
- Encode categorical features using OrdinalEncoder and OneHotEncoder
- Apply feature scaling using StandardScaler, MinMaxScaler and RobustScaler
- Apply all preprocessing steps after the train-test split to avoid data leakage

In [1]:
%pip install palmerpenguins

Note: you may need to restart the kernel to use updated packages.


## Step 1: Load and explore the dataset

In [3]:
import pandas as pd
import numpy as np
from palmerpenguins import load_penguins
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load the Palmer Penguins dataset
df_raw = load_penguins()

# Select relevant columns and drop year (not a meaningful feature)
df = df_raw[['bill_length_mm', 'flipper_length_mm', 'body_mass_g',
             'island', 'sex', 'species']].copy()

# Add a synthetic ordinal column to illustrate ordinal encoding
np.random.seed(42)
df['size_category'] = np.random.choice(
    ['Small', 'Medium', 'Large', 'Extra Large'],
    size=len(df), p=[0.20, 0.35, 0.30, 0.15])

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Dataset shape: (344, 7)

First 5 rows:


,bill_length_mm,flipper_length_mm,body_mass_g,island,sex,species,size_category
0,39.1,181.0,3750.0,Torgersen,male,Adelie,Medium
1,39.5,186.0,3800.0,Torgersen,female,Adelie,Extra Large
2,40.3,195.0,3250.0,Torgersen,female,Adelie,Large
3,NaN,NaN,NaN,Torgersen,NaN,Adelie,Large
4,36.7,193.0,3450.0,Torgersen,female,Adelie,Small


In [4]:
# Inspect data types and missing values
print("Data types:")
print(df.dtypes)
print("\n" + "="*50)
print("\nMissing values:")
for col in df.columns:
    count = df[col].isnull().sum()
    pct   = count / len(df) * 100
    status = f"{count} missing ({pct:.1f}%)" if count > 0 else "No missing values ✓"
    print(f"  {col}: {status}")

Data types:
bill_length_mm       float64
flipper_length_mm    float64
body_mass_g          float64
island                object
sex                   object
species               object
size_category         object
dtype: object


Missing values:
  bill_length_mm: 2 missing (0.6%)
  flipper_length_mm: 2 missing (0.6%)
  body_mass_g: 2 missing (0.6%)
  island: No missing values ✓
  sex: 11 missing (3.2%)
  species: No missing values ✓
  size_category: No missing values ✓


### Understanding our features

| Feature | Type | Notes | Preprocessing needed |
|---------|------|-------|----------------------|
| **bill_length_mm** | Numeric | Has missing values | Impute → Scale |
| **flipper_length_mm** | Numeric | Has missing values | Impute → Scale |
| **body_mass_g** | Numeric | Very wide range, has missing values | Impute → Scale |
| **size_category** | Ordinal | Small < Medium < Large < Extra Large | Ordinal Encoding |
| **island** | Nominal | No natural order, no missing values | One-Hot Encoding |
| **sex** | Nominal | No natural order, has missing values | Impute → One-Hot Encoding |
| **species** | Target | Adelie / Chinstrap / Gentoo | Label Encoding |

> **Note:** `bill_length_mm`, `flipper_length_mm`, `body_mass_g`, `island`, `sex` and `species` are real measurements from the Palmer Penguins dataset (Antarctica, 2007–2009). `size_category` is a synthetic column added to illustrate ordinal encoding.

## Step 2: Split first

Before any preprocessing, we split the data. This is the golden rule – any transformer that learns from data must see only the training set.

In [5]:
# Separate features and target
X = df.drop('species', axis=1)
y_raw = df['species']

# Encode target: Adelie → 0, Chinstrap → 1, Gentoo → 2
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print("Target encoding:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls} → {i}")

# Split — stratify preserves the species distribution in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining set: {X_train.shape[0]} rows")
print(f"Test set:     {X_test.shape[0]} rows")

Target encoding:
  Adelie → 0
  Chinstrap → 1
  Gentoo → 2

Training set: 275 rows
Test set:     69 rows


## Step 3: Handling missing values with SimpleImputer

Before we can scale or encode, we must handle missing values — scalers silently propagate `NaN`, producing corrupted output — impute first to avoid this. We handle numeric and categorical missing values separately.

**The golden rule applies here too:** fit the imputer on training data only, then use the same learned values to fill missing values in the test set.

In [6]:
# Define numeric columns and check missing values
numeric_cols = ['bill_length_mm', 'flipper_length_mm', 'body_mass_g']

print("Missing values in training set:")
print(X_train[numeric_cols + ['sex']].isnull().sum())
print("\nMissing values in test set:")
print(X_test[numeric_cols + ['sex']].isnull().sum())

Missing values in training set:
bill_length_mm        2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

Missing values in test set:
bill_length_mm       0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


In [7]:
# ── Numeric imputation ───────────────────────────────────────────────────
num_imputer = SimpleImputer(strategy='median')

# fit: learns the median of each column FROM TRAINING DATA ONLY
num_imputer.fit(X_train[numeric_cols])

print("Learned fill values (from training data only):")
for col, val in zip(numeric_cols, num_imputer.statistics_):
    print(f"  {col}: {val:.2f}")

# Show missing counts BEFORE transforming
print(f"\nMissing values before imputation: {X_train[numeric_cols].isnull().sum().sum()}")

# transform: fills missing values using the learned medians
X_train_num_imp = num_imputer.transform(X_train[numeric_cols])
X_test_num_imp  = num_imputer.transform(X_test[numeric_cols])

print(f"Missing values after imputation:  {pd.DataFrame(X_train_num_imp).isnull().sum().sum()}")

Learned fill values (from training data only):
  bill_length_mm: 44.90
  flipper_length_mm: 197.00
  body_mass_g: 4050.00

Missing values before imputation: 6
Missing values after imputation:  0


In [8]:
# ── Categorical imputation ───────────────────────────────────────────────
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_imputer.fit(X_train[['sex']])

print("Most frequent sex value (learned from training data):")
print(f"  sex: '{cat_imputer.statistics_[0]}'")

# Show missing count BEFORE transforming
print(f"\nMissing values before imputation: {X_train['sex'].isnull().sum()}")

# transform: fills missing values using the learned mode
X_train_sex_imp = cat_imputer.transform(X_train[['sex']]).astype(object)
X_test_sex_imp  = cat_imputer.transform(X_test[['sex']]).astype(object)

print(f"Missing values after imputation:  {pd.Series(X_train_sex_imp.flatten()).isnull().sum()}")

Most frequent sex value (learned from training data):
  sex: 'male'

Missing values before imputation: 11
Missing values after imputation:  0


### Four imputation strategies

| Strategy | Fill value | Best for |
|----------|-----------|----------|
| `'mean'` | Column average | Normally distributed numeric data, no outliers |
| `'median'` | Middle value | Numeric data with outliers or skewed distributions |
| `'most_frequent'` | Most common value | Categorical or binary features |
| `'constant'` | A fixed value you specify | When a specific fill value is meaningful (e.g. 0 or 'Unknown') |

**Why median for numeric features here?** Penguin measurements such as body mass can have outliers. The median is more robust than the mean in such cases.

## Step 4: Feature scaling

Now that missing values have been filled, we can safely scale the numeric features. Our three features have very different ranges:
- `bill_length_mm`: 32 - 60 mm
- `flipper_length_mm`: 172 - 231 mm
- `body_mass_g`: 2700 - 6300 g

Without scaling, `body_mass_g` will completely dominate any distance-based calculation. We use the imputed arrays `X_train_num_imp` and `X_test_num_imp` from Step 3 as input.

In [9]:
# Show feature ranges after imputation
X_train_num_df = pd.DataFrame(X_train_num_imp, columns=numeric_cols)

print("Feature ranges (training set, after imputation):")
print("="*60)
for col in numeric_cols:
    print(f"  {col:20s}: min={X_train_num_df[col].min():.1f}, "
          f"max={X_train_num_df[col].max():.1f}, "
          f"mean={X_train_num_df[col].mean():.1f}")

Feature ranges (training set, after imputation):
  bill_length_mm      : min=33.1, max=59.6, mean=44.1
  flipper_length_mm   : min=172.0, max=231.0, mean=200.8
  body_mass_g         : min=2700.0, max=6300.0, mean=4210.1


In [10]:
# ── StandardScaler ──────────────────────────────────────────────────────
# Input: imputed arrays from Step 3
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train_num_imp)  # fit + transform on training data
X_test_std  = std_scaler.transform(X_test_num_imp)        # transform only on test data

print("StandardScaler - centres to mean=0, std=1:")
print("="*60)
for i, col in enumerate(numeric_cols):
    print(f"  {col:20s}: mean={X_train_std[:, i].mean():.3f}, "
          f"std={X_train_std[:, i].std():.3f}")

StandardScaler - centres to mean=0, std=1:
  bill_length_mm      : mean=0.000, std=1.000
  flipper_length_mm   : mean=0.000, std=1.000
  body_mass_g         : mean=0.000, std=1.000


In [11]:
# ── MinMaxScaler ────────────────────────────────────────────────────────
mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train_num_imp)
X_test_mm  = mm_scaler.transform(X_test_num_imp)

print("MinMaxScaler - squeezes values into [0, 1]:")
print("="*50)
for i, col in enumerate(numeric_cols):
    print(f"  {col}: min={X_train_mm[:, i].min():.3f}, max={X_train_mm[:, i].max():.3f}")

# ── RobustScaler ─────────────────────────────────────────────────────────
rob_scaler = RobustScaler()
X_train_rob = rob_scaler.fit_transform(X_train_num_imp)
X_test_rob  = rob_scaler.transform(X_test_num_imp)

print("\nRobustScaler - uses median and IQR (resistant to outliers):")
print("="*50)
print(f"  Learned medians:  {rob_scaler.center_.round(1)}")
print(f"  Learned IQR:      {rob_scaler.scale_.round(1)}")

MinMaxScaler - squeezes values into [0, 1]:
  bill_length_mm: min=0.000, max=1.000
  flipper_length_mm: min=0.000, max=1.000
  body_mass_g: min=0.000, max=1.000

RobustScaler - uses median and IQR (resistant to outliers):
  Learned medians:  [  44.9  197.  4050. ]
  Learned IQR:      [   9.1   23.  1212.5]


In [12]:
# Summary comparison
X_train_num_df = pd.DataFrame(X_train_num_imp, columns=numeric_cols)

comparison = pd.DataFrame({
    'Feature':         numeric_cols,
    'Original mean':   X_train_num_df.mean().round(1).values,
    'StandardScaler':  ['mean=0, std=1'] * 3,
    'MinMaxScaler':    ['min=0, max=1'] * 3,
    'RobustScaler':    ['median=0, IQR-scaled'] * 3
})
print(comparison.to_string(index=False))

print("\nWhich to choose?")
print("  Outliers present?    -> RobustScaler")
print("  Need values in 0-1?  -> MinMaxScaler")
print("  Otherwise            -> StandardScaler (default)")

          Feature  Original mean StandardScaler MinMaxScaler         RobustScaler
   bill_length_mm           44.1  mean=0, std=1 min=0, max=1 median=0, IQR-scaled
flipper_length_mm          200.8  mean=0, std=1 min=0, max=1 median=0, IQR-scaled
      body_mass_g         4210.1  mean=0, std=1 min=0, max=1 median=0, IQR-scaled

Which to choose?
  Outliers present?    -> RobustScaler
  Need values in 0-1?  -> MinMaxScaler
  Otherwise            -> StandardScaler (default)


## Step 5: Encoding categorical features

We have two types of categorical features:
- **Ordinal**: `size_category` — has a clear order: Small < Medium < Large < Extra Large
- **Nominal**: `island`, `sex` — no natural order between categories

### Ordinal encoding for size category

We specify the order explicitly. The encoder converts each level to a number that preserves the progression.

In [13]:
# Ordinal encoding for size_category
ord_encoder = OrdinalEncoder(
    categories=[['Small', 'Medium', 'Large', 'Extra Large']]
)

# Fit on training data only
ord_encoder.fit(X_train[['size_category']])

# Transform both sets using the same learned mapping
X_train_size_enc = ord_encoder.transform(X_train[['size_category']])
X_test_size_enc  = ord_encoder.transform(X_test[['size_category']])

print("Ordinal encoding mapping:")
for level, num in zip(['Small', 'Medium', 'Large', 'Extra Large'], [0, 1, 2, 3]):
    print(f"  {level:12s} → {num}")

print(f"\nEncoded shape (train): {X_train_size_enc.shape}")
print(f"Encoded shape (test):  {X_test_size_enc.shape}")

Ordinal encoding mapping:
  Small        → 0
  Medium       → 1
  Large        → 2
  Extra Large  → 3

Encoded shape (train): (275, 1)
Encoded shape (test):  (69, 1)


### One-hot encoding for nominal features

`island` and `sex` have no natural order — we give each category its own binary column.

In [ ]:
# One-hot encoding for island and sex
# sex has missing values so we use the imputed arrays from Step 3

X_train_nom = np.hstack([
    X_train[['island']].values.astype(object),
    X_train_sex_imp   # imputed in the categorical imputation step above
])
X_test_nom = np.hstack([
    X_test[['island']].values.astype(object),
    X_test_sex_imp    # imputed in the categorical imputation step above
])

# Fit on training data only
ohe = OneHotEncoder(handle_unknown='ignore')
ohe.fit(X_train_nom)

# Transform both sets using the same learned mapping
X_train_ohe = ohe.transform(X_train_nom)
X_test_ohe  = ohe.transform(X_test_nom)

print("Encoded column names:")
print(ohe.get_feature_names_out(['island', 'sex']))
print(f"\nOne-hot encoded shape (train): {X_train_ohe.toarray().shape}")
print(f"One-hot encoded shape (test):  {X_test_ohe.toarray().shape}")

Encoded column names:
['island_Biscoe' 'island_Dream' 'island_Torgersen' 'sex_female' 'sex_male']

One-hot encoded shape (train): (275, 5)
One-hot encoded shape (test):  (69, 5)


### Unseen categories

What happens if test data contains a category the encoder never saw during training?

`handle_unknown='ignore'` encodes it as all zeros – the model treats it as "none of the known categories". This prevents crashes in production.

In [15]:
# Preview the one-hot encoded output
ohe_df = pd.DataFrame(
    X_train_ohe.toarray(),
    columns=ohe.get_feature_names_out(['island', 'sex'])
)
print("Sample of one-hot encoded output (first 5 rows):")
print(ohe_df.head())

Sample of one-hot encoded output (first 5 rows):
   island_Biscoe  island_Dream  island_Torgersen  sex_female  sex_male
0            0.0           1.0               0.0         1.0       0.0
1            1.0           0.0               0.0         1.0       0.0
2            0.0           0.0               1.0         1.0       0.0
3            1.0           0.0               0.0         0.0       1.0
4            1.0           0.0               0.0         0.0       1.0


## Exercise: Apply what you've learned

A new column has been added to the dataset: `bmi_category` with values `'Underweight'`, `'Healthy'`, `'Overweight'`, `'Obese'`.

**Your task:**
1. Decide whether `bmi_category` is nominal or ordinal.
2. Choose the correct encoding method and explain why.
3. Apply the encoding correctly — remember to fit on training data only.
4. Print the first 8 rows showing the original values and their encoded equivalents.

In [16]:
# Add the new column to the dataset
np.random.seed(42)
X_train_new = X_train.copy()
X_test_new  = X_test.copy()

X_train_new['bmi_category'] = np.random.choice(
    ['Underweight', 'Healthy', 'Overweight', 'Obese'], len(X_train),
    p=[0.10, 0.45, 0.30, 0.15])
X_test_new['bmi_category'] = np.random.choice(
    ['Underweight', 'Healthy', 'Overweight', 'Obese'], len(X_test),
    p=[0.10, 0.45, 0.30, 0.15])

print("New column added:")
print(X_train_new['bmi_category'].value_counts())

New column added:
bmi_category
Healthy        121
Overweight      78
Obese           47
Underweight     29
Name: count, dtype: int64


<details>
<summary>Select to reveal solution (try it yourself first!)</summary>

`bmi_category` is **ordinal** — there is a clear progression: Underweight < Healthy < Overweight < Obese.
Use `OrdinalEncoder` and specify the order explicitly.

```python
bmi_encoder = OrdinalEncoder(
    categories=[['Underweight', 'Healthy', 'Overweight', 'Obese']]
)

# Fit on training data only
bmi_encoder.fit(X_train_new[['bmi_category']])

# Transform both sets using the same learned mapping
train_bmi_encoded = bmi_encoder.transform(X_train_new[['bmi_category']])
test_bmi_encoded  = bmi_encoder.transform(X_test_new[['bmi_category']])
```

</details>

In [1]:
# Solution
bmi_encoder = OrdinalEncoder(
    categories=[['Underweight', 'Healthy', 'Overweight', 'Obese']]
)

# Fit on training data only
bmi_encoder.fit(X_train_new[['bmi_category']])

# Transform both sets using the same learned mapping
train_bmi_encoded = bmi_encoder.transform(X_train_new[['bmi_category']])
test_bmi_encoded  = bmi_encoder.transform(X_test_new[['bmi_category']])

print("Encoding mapping:")
for level, num in zip(['Underweight', 'Healthy', 'Overweight', 'Obese'], [0, 1, 2, 3]):
    print(f"  {level:12s} → {num}")

print("\nFirst 8 rows — original vs encoded:")
for original, encoded in zip(
    X_train_new['bmi_category'].values[:8],
    train_bmi_encoded.flatten()[:8]
):
    print(f"  {original:12s} → {encoded:.0f}")

NameError: name 'OrdinalEncoder' is not defined

## Summary

### The preprocessing steps covered in this notebook

```
Raw data
    │
    ├─ Split first (train_test_split)
    │
    ├─ Numeric columns
    │      SimpleImputer(strategy='median')  → fit on train, transform train + test
    │      StandardScaler()                  → fit on train, transform train + test
    │
    ├─ Nominal columns
    │      SimpleImputer(strategy='most_frequent')       → fit on train, transform train + test
    │      OneHotEncoder(handle_unknown='ignore')        → fit on train, transform train + test
    │
    └─ Ordinal columns
           OrdinalEncoder(categories=[explicit_order])  → fit on train, transform train + test
```

### Key takeaways

1. Split before fitting: imputers, scalers and encoders must learn from training data only.
2. Impute before scaling/encoding: NaN propagates silently through sklearn transformers.
3. Scale numeric features: so no single feature dominates by range.
4. Ordinal vs nominal matters: one-hot on ordinal data loses ordering information.
5. Always use `handle_unknown='ignore'` on OneHotEncoder for production robustness.

### What's next
Lesson 4 shows how to combine all these steps into a single Pipeline and ColumnTransformer, making the workflow cleaner and leakage-proof automatically.